# 01. 定义add算子（Ascend版本）

In [ ]:
import tilelang
import tilelang.language as T


M, N = 1024, 1024
block_M, block_N = 128, 256
dtype = "float"

@tilelang.jit(out_idx=[-1], pass_configs={
    tilelang.PassConfigKey.TL_ASCEND_AUTO_SYNC: True,
})
def my_add():
    m_num = M // block_M
    n_num = N // block_N

    VEC_NUM = 2

    @T.prim_func
    def main(
        A: T.Tensor(shape=(M, N), dtype=dtype), # type: ignore
        B: T.Tensor(shape=(M, N), dtype=dtype), # type: ignore
        C: T.Tensor(shape=(M, N), dtype=dtype), # type: ignore
    ):
        with T.Kernel(m_num * n_num, is_npu = True) as (cid, vid):
            bx = cid // n_num
            by = cid % n_num
            
            a_ub = T.alloc_ub((block_M // VEC_NUM, block_N), dtype)
            b_ub = T.alloc_ub((block_M // VEC_NUM, block_N), dtype)
            c_ub = T.alloc_ub((block_M // VEC_NUM, block_N), dtype)
            with T.Scope("V"):
                T.copy(A[bx * block_M + vid * block_M // VEC_NUM, by * block_N], a_ub)
                T.copy(B[bx * block_M + vid * block_M // VEC_NUM, by * block_N], b_ub)
                T.tile.add(c_ub, a_ub, b_ub)
                T.copy(c_ub, C[bx * block_M + vid * block_M // VEC_NUM, by * block_N])
    return main

my_jit_add = my_add()
# print(my_jit_add.get_kernel_source())

# 02. 运行&验证

In [ ]:
import torch

torch.manual_seed(0)

a = torch.randn(M, N).npu()
b = torch.randn(M, N).npu()
c_golden = a + b

# print(f"Input tensor a: {a}")
# print(f"Input tensor b: {b}")
# print(f"Golden tensor c: {c_golden}")

c = my_jit_add(a, b)
# print(f"Output tensor c: {c}")

torch.testing.assert_close(c, c_golden, rtol=1e-5, atol=1e-5)
print("Test passed: The output tensor c is equal to the sum of the input tensors a and b.")
